# 09c — Soluzioni: libreria client OpenAI

**Corso**: Programmazione di Applicazioni Intelligenti  
**Lezione 09** — Dal Client OpenAI al Function Calling e MCP  
**Blocco 2** — Soluzioni esercitazione

Questo notebook contiene le soluzioni commentate degli esercizi del notebook `09b`.


---
## Setup


In [ ]:
# Setup
!pip install -q openai

from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY"),
)

MODEL = "openai/gpt-oss-120b"
print(f"Client pronto! Modello: {MODEL}")


---
## Esercizio 1 — Setup e prima chiamata (soluzione)


In [ ]:
# Esercizio 1 — Soluzione

# Il system prompt definisce la "personalità" del modello.
# Tutto ciò che scriviamo qui influenzerà tono, stile e vincoli delle risposte.
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "Sei un pirata esperto di tecnologia. Rispondi sempre in italiano e con il tono di un pirata: usa espressioni come 'Arr!', 'ciurma', 'tesoro', ecc."
        },
        {
            "role": "user",
            "content": "Cos'è Python?"
        }
    ]
)

# La risposta è in choices[0] perché n=1 (default)
print("=== RISPOSTA ===")
print(response.choices[0].message.content)
print()

# Il campo usage ci dice quanti token abbiamo consumato
print("=== TOKEN ===")
print(f"Prompt (input):     {response.usage.prompt_tokens}")
print(f"Completion (output): {response.usage.completion_tokens}")
print(f"Totali:              {response.usage.total_tokens}")


---
## Esercizio 2 — Chatbot multi-turno (soluzione)

La chiave è mantenere una lista `cronologia` che cresce ad ogni turno. Ad ogni chiamata API, inviamo *tutta* la cronologia — è così che il modello "ricorda" i turni precedenti.


In [ ]:
# Esercizio 2 — Soluzione

system_prompt = {"role": "system", "content": "Sei un assistente amichevole. Rispondi in italiano, in modo conciso."}

# La cronologia parte con il solo system prompt
cronologia = [system_prompt]

print("Chatbot avviato! Comandi: 'esci' per uscire, '/reset' per ricominciare.")
print()

while True:
    user_input = input("Tu: ")

    if user_input.lower() == "esci":
        print("Arrivederci!")
        break

    # Bonus: /reset azzera la cronologia ma mantiene il system prompt
    if user_input.strip() == "/reset":
        cronologia = [system_prompt]
        print("--- Cronologia azzerata! ---")
        print()
        continue

    # Aggiungiamo il messaggio dell'utente alla cronologia
    cronologia.append({"role": "user", "content": user_input})

    # Chiamata API con tutta la cronologia
    response = client.chat.completions.create(
        model=MODEL,
        messages=cronologia,
    )

    # Estraiamo la risposta
    risposta = response.choices[0].message.content

    # Aggiungiamo la risposta dell'assistente alla cronologia
    # Così al prossimo turno il modello "ricorderà" anche questa risposta
    cronologia.append({"role": "assistant", "content": risposta})

    print(f"Assistente: {risposta}")
    print(f"  [token totali in questa chiamata: {response.usage.total_tokens} | messaggi in cronologia: {len(cronologia)}]")
    print()


---
## Esercizio 3 — Output strutturato con Pydantic (soluzione)

Definiamo la classe Pydantic come schema e usiamo `client.beta.chat.completions.parse()` per ottenere direttamente un oggetto Python tipizzato.


In [ ]:
# Esercizio 3 — Soluzione

from pydantic import BaseModel
from typing import List

# Lo schema Pydantic descrive esattamente la struttura che vogliamo.
# Il modello è obbligato (via constrained decoding) a rispettarla.
class CityInfo(BaseModel):
    name: str
    country: str
    population: int
    famous_for: List[str]
    is_capital: bool


citta = ["Roma", "Tokyo", "New York"]
risultati = []

for nome_citta in citta:
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Fornisci informazioni sulla città richiesta. Popolazione approssimativa."
            },
            {
                "role": "user",
                "content": f"Dammi le informazioni su {nome_citta}."
            }
        ],
        response_format=CityInfo,
    )

    # .parsed restituisce direttamente un oggetto CityInfo, non una stringa
    info = response.choices[0].message.parsed
    risultati.append(info)
    print(f"Analizzata: {info.name} ({info.country})")

print(f"\nRaccolte {len(risultati)} città.")


In [ ]:
# Tabella riepilogativa
print(f"{'Città':<15} {'Paese':<15} {'Popolazione':<15} {'Capitale':<10} {'Famosa per'}")
print("-" * 90)

for info in risultati:
    # Tronchiamo la lista famous_for per la visualizzazione
    famosa = ", ".join(info.famous_for[:3])
    if len(info.famous_for) > 3:
        famosa += ", ..."
    cap = "Sì" if info.is_capital else "No"
    print(f"{info.name:<15} {info.country:<15} {info.population:<15,} {cap:<10} {famosa}")


In [ ]:
# Possiamo anche convertire tutto in una lista di dizionari
# utile per esportare in CSV, JSON, DataFrame, ecc.
import json
dati = [info.model_dump() for info in risultati]
print(json.dumps(dati, indent=2, ensure_ascii=False))


---
## Esercizio 4 (Bonus) — Reasoning visibile (soluzione)

Confrontiamo la stessa domanda con e senza reasoning per osservare come il modello "pensa".


In [ ]:
# Esercizio 4 (Bonus) — Soluzione

domanda = "Se ho 3 scatole e in ogni scatola ci sono 4 sacchetti, e in ogni sacchetto ci sono 5 biglie, quante biglie ho in totale?"

# --- Senza reasoning ---
response_base = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": domanda}],
)

print("=== SENZA REASONING ===")
print(f"Risposta: {response_base.choices[0].message.content}")
print(f"Token totali: {response_base.usage.total_tokens}")
print()

# --- Con reasoning ---
response_reasoning = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": domanda}],
    reasoning_effort="high",
)

msg = response_reasoning.choices[0].message

print("=== CON REASONING ===")

# Su Groq, gpt-oss-120b espone il reasoning nel campo message.reasoning
if hasattr(msg, 'reasoning') and msg.reasoning:
    print(f"Reasoning (primi 800 caratteri):")
    print(msg.reasoning[:800])
    if len(msg.reasoning) > 800:
        print("...")
    print()

print(f"Risposta: {msg.content}")
print(f"Token totali: {response_reasoning.usage.total_tokens}")

# --- Confronto ---
diff = response_reasoning.usage.total_tokens - response_base.usage.total_tokens
print(f"\n=== CONFRONTO ===")
print(f"Token extra con reasoning: {diff} (+{diff / response_base.usage.total_tokens * 100:.0f}%)")
print(f"Il reasoning consuma più token ma può produrre risposte più accurate,")
print(f"soprattutto per problemi che richiedono ragionamento step-by-step.")


---
## Riepilogo

In questa esercitazione hai praticato:
- **System prompt** per modificare il comportamento del modello
- **Multi-turno** con cronologia manuale e osservazione della crescita dei token
- **Structured output** con Pydantic per ottenere dati tipizzati dal modello
- **Reasoning** per vedere come il modello "pensa" prima di rispondere

Nel prossimo blocco (Notebook 09d) daremo **superpoteri** all'LLM con il **function calling** e il protocollo **MCP**.
